In [1]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

### Objectives:-
1. Read car sales data as csv
2. Perform data cleaning by checking for nulls and dropping unwanted columns 
3. Perform feature encoding using LabelEncoder, OneHotEncoder, and StandardScaler
4. Train, test, split data
5. Train multiple Regression models (LinearRegression, Ridge, Lasso, RandomForestRegressor, KNeighborsRegressor, DecisionTreeRegressor)
6. Print and review performance metrics (r2_score, mean_absolute_error, mean_squared_error) for both test and training data
7. Take two models with the best performance metrics (RandomForestRegressor, KNeighborsRegressor) and fine tune them by passing hyperparameters using RandomizedSearchCV
8. Find best params
9. Rerun RandomForestRegressor and KNeighborsRegressor models with the best parameters
10. Print and review performance metrics (r2_score, mean_absolute_error, mean_squared_error) for both test and training data 


In [2]:
df = pd.read_csv('../data/raw/cardekho_imputated.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15411 entries, 0 to 15410
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         15411 non-null  int64  
 1   car_name           15411 non-null  str    
 2   brand              15411 non-null  str    
 3   model              15411 non-null  str    
 4   vehicle_age        15411 non-null  int64  
 5   km_driven          15411 non-null  int64  
 6   seller_type        15411 non-null  str    
 7   fuel_type          15411 non-null  str    
 8   transmission_type  15411 non-null  str    
 9   mileage            15411 non-null  float64
 10  engine             15411 non-null  int64  
 11  max_power          15411 non-null  float64
 12  seats              15411 non-null  int64  
 13  selling_price      15411 non-null  int64  
dtypes: float64(2), int64(6), str(6)
memory usage: 1.6 MB


### Data Cleaning

In [3]:
df.drop('Unnamed: 0', axis=1, inplace=True)
df.head()

,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [4]:
df.isnull().sum()

car_name             0
brand                0
model                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

In [5]:
# out 0f drop car_name and brand and model let's drop 2 columns as it's repetitive infomration

df.drop(['brand', 'car_name'], axis=1, inplace=True)
df.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [6]:
# split data into dependent and independent features
X, y = df.drop('selling_price', axis=1), df['selling_price']

### Feature Encoding 

In [7]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

label_encoder = LabelEncoder()
X['model'] = label_encoder.fit_transform(X['model'])

In [8]:
# Create Column transformer
from sklearn.compose import ColumnTransformer

num_feat = X.select_dtypes(exclude='str').columns
one_hot_columns = ['seller_type', 'fuel_type', 'transmission_type']

num_transformer = StandardScaler()
one_hot_transformer = OneHotEncoder(drop='first')

preprocessor = ColumnTransformer([("text_preprocess", one_hot_transformer, one_hot_columns),("num_preprocess", num_transformer, num_feat)],
                                  remainder='passthrough')

X = preprocessor.fit_transform(X)

In [9]:
# train, test, split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [10]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV

In [11]:
def evaluate_model(true_val, predicted_value):
    mse = mean_squared_error(true_val, predicted_value)
    mae = mean_absolute_error(true_val, predicted_value)
    r2 = r2_score(true_val, predicted_value)

    return mse, mae, r2


In [12]:
# Let's test multiple linear regression model we have learned so far

models = {'Random Forest Classifier': RandomForestRegressor(), 'Linear Regression': LinearRegression(), 'Ridge': Ridge(), 'Lasso': Lasso(),
          'K Neighbors Regressor': KNeighborsRegressor(), 'Decision Tree Regressor': DecisionTreeRegressor()}

for k, v in models.items():
    model = v
    model.fit(X_train, y_train)

    # make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    mse_train, mae_train, r2_train = evaluate_model(y_train, y_train_pred)
    mse_test, mae_test, r2_test = evaluate_model(y_test, y_test_pred)

    print('===========================')
    print(k,':- Metrics')
    print('Train Model Performance Metrics ')
    print(f'root_mean_squared_error:- {mse_train}')
    print(f'mean_absolute_error:- {mae_train}')
    print(f'r2_score:- {r2_train}')

    print('-------------------')

    print('\n')

    print('Test Model Performance Metrics ')
    print(f'root_mean_squared_error:- {mse_test}')
    print(f'mean_absolute_error:- {mae_test}')
    print(f'r2_score:- {r2_test}')




Random Forest Classifier :- Metrics
Train Model Performance Metrics 
root_mean_squared_error:- 14544728727.27916
mean_absolute_error:- 39766.50929553448
r2_score:- 0.9820664551279484
-------------------


Test Model Performance Metrics 
root_mean_squared_error:- 50850403939.61679
mean_absolute_error:- 100970.12818929549
r2_score:- 0.9324500079804321
Linear Regression :- Metrics
Train Model Performance Metrics 
root_mean_squared_error:- 306756099359.7596
mean_absolute_error:- 268101.6070829937
r2_score:- 0.6217719576765959
-------------------


Test Model Performance Metrics 
root_mean_squared_error:- 252550062888.5656
mean_absolute_error:- 279618.57941584283
r2_score:- 0.6645109298852004
Ridge :- Metrics
Train Model Performance Metrics 
root_mean_squared_error:- 306756818740.9266
mean_absolute_error:- 268059.8014688303
r2_score:- 0.6217710706848424
-------------------


Test Model Performance Metrics 
root_mean_squared_error:- 252540243247.9684
mean_absolute_error:- 279557.2168930266
r

In [13]:
# Randforest and K Neighbor models have performed the best. So let's play with some parameters to to see if we can improve performance metrics further

knn_params = {'n_neighbors': [2,3,10,20,40,50]}
rf_params = {'max_depth': [None, 5,8,10, 15], 'max_features': [5,7,8, 'auto'], 'min_samples_split': [2,8,15,20], 'n_estimators': [100, 200, 500, 1000]}

In [14]:
random_cv_model = [('KNNN', KNeighborsRegressor(), knn_params), ('Random Forest Regressor', RandomForestRegressor(), rf_params)]

In [15]:
# Initialize and fit models with hyperparameters
model_param = {}

for name, model, params in random_cv_model:
    random = RandomizedSearchCV(estimator=model, param_distributions=params, n_iter=100, cv=3, verbose=2, n_jobs=-1)
    random.fit(X_train, y_train)
    model_param[name] = random.best_params_

for k, v in model_param.items():
    print(f'--------Best Params For Model {k} ------------')
    print(v)

Fitting 3 folds for each of 6 candidates, totalling 18 fits
[CV] END ......................................n_neighbors=2; total time=   0.3s
[CV] END ......................................n_neighbors=2; total time=   0.3s
[CV] END ......................................n_neighbors=2; total time=   0.3s
[CV] END ......................................n_neighbors=3; total time=   0.4s
[CV] END ......................................n_neighbors=3; total time=   0.4s
[CV] END ......................................n_neighbors=3; total time=   0.4s
[CV] END .....................................n_neighbors=10; total time=   0.6s
[CV] END .....................................n_neighbors=10; total time=   0.6s
[CV] END .....................................n_neighbors=10; total time=   0.7s
[CV] END .....................................n_neighbors=20; total time=   0.8s
[CV] END .....................................n_neighbors=20; total time=   0.8s
[CV] END .....................................n_n

In [16]:
# let's retrain our models with best params

models = {'Random Forest Classifier': RandomForestRegressor(max_depth=10, max_features=7, min_samples_split=2, n_estimators=200, n_jobs=-1),
          'K Neighbors Regressor': KNeighborsRegressor(n_neighbors=10, n_jobs=-1)}

for k, v in models.items():
    model = v
    model.fit(X_train, y_train)

    # make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    mse_train, mae_train, r2_train = evaluate_model(y_train, y_train_pred)
    mse_test, mae_test, r2_test = evaluate_model(y_test, y_test_pred)

    print('===========================')
    print(k,':- Metrics')
    print('Train Model Performance Metrics ')
    print(f'root_mean_squared_error:- {mse_train}')
    print(f'mean_absolute_error:- {mae_train}')
    print(f'r2_score:- {r2_train}')

    print('-------------------')

    print('\n')

    print('Test Model Performance Metrics ')
    print(f'root_mean_squared_error:- {mse_test}')
    print(f'mean_absolute_error:- {mae_test}')
    print(f'r2_score:- {r2_test}')


Random Forest Classifier :- Metrics
Train Model Performance Metrics 
root_mean_squared_error:- 30469939116.875244
mean_absolute_error:- 84126.54533615462
r2_score:- 0.9624307863936777
-------------------


Test Model Performance Metrics 
root_mean_squared_error:- 50902177233.605255
mean_absolute_error:- 106315.19723323057
r2_score:- 0.9323812320155469
K Neighbors Regressor :- Metrics
Train Model Performance Metrics 
root_mean_squared_error:- 132092099887.24854
mean_absolute_error:- 103456.64341336794
r2_score:- 0.8371314003176575
-------------------


Test Model Performance Metrics 
root_mean_squared_error:- 69620968665.46384
mean_absolute_error:- 117432.89815115147
r2_score:- 0.9075150733643885
